In [27]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [28]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍


In [29]:
import subprocess
from IPython.display import HTML, display
import html

cmd = f"""
cd ../../firmware/mcu/simpleserial-xor
make PLATFORM={PLATFORM} CRYPTO_TARGET=NONE SS_VER={SS_VER} -j
"""

result = subprocess.run( cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

display(HTML(f"""
<div style="max-height: 300px; overflow-y: auto; border: 1px solid #888; padding: 8px; background-color: #111; color: #ddd; font-family: monospace;
    white-space: pre-wrap;">
{html.escape(result.stdout)}
</div>
"""))

In [30]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-xor/simpleserial-xor-{}.hex".format(PLATFORM))

In [32]:
def hexstr(data):
    return " ".join(f"{b:02X}" for b in data)

In [33]:
ktp = cw.ktp.Basic()
key, text = ktp.next()

target.flush()

target.simpleserial_write('b', key)
ack = target.simpleserial_wait_ack()

target.simpleserial_write('a', text)

response = target.simpleserial_read('r', 16)

expected = bytearray(t ^ k for t, k in zip(text, key))

print("received:", hexstr(response))
print("expected:", hexstr(expected))
print("match   :", response == expected)

received: 2D 34 B9 37 B1 13 76 A1 70 8A D6 29 A5 83 3A 60
expected: 2D 34 B9 37 B1 13 76 A1 70 8A D6 29 A5 83 3A 60
match   : True
